In [0]:
%sql
CREATE OR REPLACE TABLE mashup_learning.stocks.silver_daily_prices
USING DELTA
AS
SELECT 
  CAST(date AS DATE) AS trade_date,
  LAG(close, 1, NULL) OVER (PARTITION BY ticker ORDER BY date) AS previous_day_close,
  CASE 
    WHEN previous_day_close IS NULL THEN NULL
    WHEN previous_day_close = 0 THEN NULL
    ELSE (close - previous_day_close) / previous_day_close * 100 
  END AS daily_return,
  CASE 
    WHEN COUNT(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) < 7 THEN NULL 
    ELSE AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) 
  END AS rolling_avg_7d,
  open, high, low, close, volume, dividends, stock_splits, ticker, source, ingested_at
FROM mashup_learning.stocks.bronze_daily_prices;

In [0]:
%sql
SELECT COUNT(*) FROM mashup_learning.stocks.silver_daily_prices;



In [0]:
%sql
SELECT 
  SUM(CASE WHEN daily_return IS NULL THEN 1 ELSE 0 END) AS null_daily_return_count,
  SUM(CASE WHEN rolling_avg_7d IS NULL THEN 1 ELSE 0 END) AS null_rolling_avg_count
FROM mashup_learning.stocks.silver_daily_prices;